# Forward runner threat and training-only calibration

This notebook executes the frozen monthly pipeline, checks temporal provenance, and reviews calibration. 2024 is previously inspected development data. 2025 stays closed. Game 716404 is quarantined because its reported start follows completion; the original baseline remains preserved. See `docs/forward_threat_protocol.md`.

In [1]:
from pathlib import Path
import sys, json, hashlib
import pandas as pd
import numpy as np
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
from scripts.forward_threat import main, OUT, LOCAL
completed = main()
OUT, LOCAL = completed.results, completed.local

2022-07-01 21912 insufficient_prior_attempts_or_nonattempts
2022-08-01 24215 logistic_recalibration
2022-09-01 22089 logistic_recalibration
2022-10-01 4794 logistic_recalibration
2023-03-01 908 insufficient_prior_attempts_or_nonattempts
2023-04-01 23694 insufficient_prior_attempts_or_nonattempts
2023-05-01 23080 logistic_recalibration
2023-06-01 22015 logistic_recalibration
2023-07-01 20995 logistic_recalibration
2023-08-01 23803 logistic_recalibration
2023-09-01 23234 logistic_recalibration
2023-10-01 1072 logistic_recalibration
{
  "selected_policy": "identity",
  "selection_raw": {
    "log_loss": 0.07157454825156918,
    "brier": 0.015990823995626204,
    "average_precision": 0.0890666136091793
  },
  "selection_calibrated": {
    "log_loss": 0.07161163400475788,
    "brier": 0.015910775689215124,
    "average_precision": 0.08878479242803214
  },
  "final_calibration_n": 138801,
  "final_calibration_attempts": 2376,
  "final_intercept": -0.27753746216864517,
  "final_slope": 0.9081

In [2]:
origins = pd.read_csv(OUT / 'origins.csv')
for row in origins.itertuples():
    assert pd.Timestamp(row.fit_max_completion) < pd.Timestamp(row.origin) <= pd.Timestamp(row.score_min_start)
    if pd.notna(row.calibration_max_completion):
        assert pd.Timestamp(row.calibration_max_completion) < pd.Timestamp(row.origin)
scores = pd.read_csv(LOCAL / 'training_forward_scores.csv.gz')
assert scores.opportunity_id.is_unique and len(scores) == 276594
assert scores.calibration_status.eq('quarantined_invalid_game_chronology').sum() == 61
assert scores.loc[scores.calibration_status.eq('quarantined_invalid_game_chronology'), 'raw'].isna().all()
assert scores.loc[scores.raw.notna(), ['raw', 'prospective_calibrated']].ge(0).all().all()
assert scores.loc[scores.raw.notna(), ['raw', 'prospective_calibrated']].le(1).all().all()
assert not (ROOT / 'data/historical_feed_manifest_2025.csv').exists()
provenance = json.loads((OUT / 'provenance.json').read_text())
assert hashlib.sha256((ROOT / 'docs/forward_threat_protocol.md').read_bytes()).hexdigest() == provenance['hashes']['protocol']
pd.read_csv(OUT / 'sample_flow.csv')

Out[2]: 
   season  scored  ... terminal  runner_change
0    2022   False  ...     1496           1335
1    2022    True  ...      470            451
2    2022    True  ...     1169            983
3    2023   False  ...        2              1
4    2023    True  ...      669            450
5    2023    True  ...     3157           2101

[6 rows x 8 columns]


,season,scored,calibration_status,n,games,attempts,terminal,runner_change
0,2022,False,warmup_no_prediction,64722,1138,844,1496,1335
1,2022,True,insufficient_prior_attempts_or_nonattempts,21912,393,261,470,451
2,2022,True,logistic_recalibration,51098,899,692,1169,983
3,2023,False,quarantined_invalid_game_chronology,61,1,1,2,1
4,2023,True,insufficient_prior_attempts_or_nonattempts,24602,425,407,669,450
5,2023,True,logistic_recalibration,114199,2004,1969,3157,2101


In [3]:
policy = json.loads((OUT / 'policy.json').read_text())
policy

Out[3]: 
{'selected_policy': 'identity',
 'selection_raw': {'log_loss': 0.07157454825156918,
  'brier': 0.015990823995626204,
  'average_precision': 0.0890666136091793},
 'selection_calibrated': {'log_loss': 0.07161163400475788,
  'brier': 0.015910775689215124,
  'average_precision': 0.08878479242803214},
 'final_calibration_n': 138801,
 'final_calibration_attempts': 2376,
 'final_intercept': -0.27753746216864517,
 'final_slope': 0.9081095891122897}


{'selected_policy': 'identity',
 'selection_raw': {'log_loss': 0.07157454825156918,
  'brier': 0.015990823995626204,
  'average_precision': 0.0890666136091793},
 'selection_calibrated': {'log_loss': 0.07161163400475788,
  'brier': 0.015910775689215124,
  'average_precision': 0.08878479242803214},
 'final_calibration_n': 138801,
 'final_calibration_attempts': 2376,
 'final_intercept': -0.27753746216864517,
 'final_slope': 0.9081095891122897}

In [4]:
metrics = pd.read_csv(OUT / 'metrics.csv')
calibration = pd.read_csv(OUT / 'calibration.csv')
subgroups = pd.read_csv(OUT / 'subgroups.csv')
for (split, model), group in calibration.groupby(['split', 'model']):
    row = metrics[(metrics.split == split) & (metrics.model == model)].iloc[0]
    assert group.n.sum() == row.n and group.attempts.sum() == row.attempts
for (split, model, axis), group in subgroups.groupby(['split', 'model', 'subgroup']):
    row = metrics[(metrics.split == split) & (metrics.model == model)].iloc[0]
    assert group.n.sum() == row.n and group.attempts.sum() == row.attempts
metrics

Out[4]: 
              split                   model  ...     brier  average_precision
0      forward_2022                     raw  ...  0.012644           0.066081
1      forward_2022  prospective_calibrated  ...  0.012615           0.065088
2      forward_2023                     raw  ...  0.016477           0.081505
3      forward_2023  prospective_calibrated  ...  0.016400           0.081017
4    selection_2023                     raw  ...  0.015991           0.089067
5    selection_2023  prospective_calibrated  ...  0.015911           0.088785
6  development_2024                     raw  ...  0.017220           0.087265
7  development_2024  prospective_calibrated  ...  0.017127           0.087265

[8 rows x 11 columns]


,split,model,n,attempts,mean_prediction,observed_rate,calibration_intercept,calibration_slope,log_loss,brier,average_precision
0,forward_2022,raw,73010,953,0.012702,0.013053,-0.171727,0.946422,0.060134,0.012644,0.066081
1,forward_2022,prospective_calibrated,73010,953,0.012587,0.013053,0.041098,1.000751,0.060154,0.012615,0.065088
2,forward_2023,raw,138801,2376,0.016631,0.017118,-0.277537,0.908110,0.073928,0.016477,0.081505
3,forward_2023,prospective_calibrated,138801,2376,0.017632,0.017118,-0.057511,0.992500,0.073877,0.016400,0.081017
4,selection_2023,raw,69104,1158,0.017453,0.016757,-0.272114,0.929894,0.071575,0.015991,0.089067
5,selection_2023,prospective_calibrated,69104,1158,0.018296,0.016757,0.016349,1.033916,0.071612,0.015911,0.088785
6,development_2024,raw,136487,2462,0.017707,0.018038,-0.409013,0.867785,0.076796,0.017220,0.087265
7,development_2024,prospective_calibrated,136487,2462,0.017997,0.018038,-0.179603,0.945808,0.076570,0.017127,0.087265


In [5]:
calibration[calibration.split.eq('development_2024')]

Out[5]: 
               split                   model  ... mean_prediction  observed_rate
42  development_2024                     raw  ...        0.001164       0.000715
43  development_2024                     raw  ...        0.007237       0.007118
44  development_2024                     raw  ...        0.014228       0.017044
45  development_2024                     raw  ...        0.030489       0.038770
46  development_2024                     raw  ...        0.068668       0.073534
47  development_2024                     raw  ...        0.137549       0.111564
48  development_2024                     raw  ...        0.294341       0.163671
49  development_2024  prospective_calibrated  ...        0.000842       0.000355
50  development_2024  prospective_calibrated  ...        0.007437       0.005725
51  development_2024  prospective_calibrated  ...        0.014336       0.014549
52  development_2024  prospective_calibrated  ...        0.030228       0.037237
53  development_202

,split,model,bin,n,attempts,mean_prediction,observed_rate
42,development_2024,raw,"(-0.001, 0.005]",46144,33,0.001164,0.000715
43,development_2024,raw,"(0.005, 0.01]",30767,219,0.007237,0.007118
44,development_2024,raw,"(0.01, 0.02]",29805,508,0.014228,0.017044
45,development_2024,raw,"(0.02, 0.05]",19706,764,0.030489,0.038770
46,development_2024,raw,"(0.05, 0.1]",6310,464,0.068668,0.073534
47,development_2024,raw,"(0.1, 0.2]",2698,301,0.137549,0.111564
48,development_2024,raw,"(0.2, 1.0]",1057,173,0.294341,0.163671
49,development_2024,prospective_calibrated,"(-0.001, 0.005]",39456,14,0.000842,0.000355
50,development_2024,prospective_calibrated,"(0.005, 0.01]",30393,174,0.007437,0.005725
51,development_2024,prospective_calibrated,"(0.01, 0.02]",33749,491,0.014336,0.014549


In [6]:
pd.read_csv(OUT / 'paired_bootstrap.csv')

Out[6]: 
              split  ... calibrated_minus_raw_upper
0    selection_2023  ...                   0.000171
1    selection_2023  ...                  -0.000042
2    selection_2023  ...                   0.000011
3  development_2024  ...                  -0.000141
4  development_2024  ...                  -0.000071
5  development_2024  ...                   0.000000

[6 rows x 4 columns]


,split,metric,calibrated_minus_raw_lower,calibrated_minus_raw_upper
0,selection_2023,log_loss,-0.000091,0.000171
1,selection_2023,brier,-0.000115,-0.000042
2,selection_2023,average_precision,-0.000663,0.000011
3,development_2024,log_loss,-0.000310,-0.000141
4,development_2024,brier,-0.000116,-0.000071
5,development_2024,average_precision,0.000000,0.000000


The historical `raw` and `prospective_calibrated` columns are generated without later fitting or calibration outcomes. The policy selected using late 2023 is applied only to 2024; do not backfill a selected-policy column over the training data. Warmup and quarantined rows have no threat score. A subsequent pitcher-response model must explicitly define its scored population, retain terminal/nonattempt windows within it, and freeze its response-stage protocol before fitting. These are predictive probabilities, not causal effects or advice about throwing over.